# Explore here

In [1]:
!pip install requests pandas matplotlib seaborn sqlalchemy

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 23.1.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [25]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

paises = ['chn', 'ago', 'ken', 'usa']
indicadores = {'SP.POP.TOTL':'Poblacion_Total', 
              #'EN.ATM.CO2E.PC': 'CO2_Per_Capita', 
              'NY.GDP.PCAP.CD':  'PIB_per_capita_(USD actuales)', 
              'SP.DYN.LE00.IN': 'Natalidad_por_1000_habitantes'}


#for ind in indicadores:
#    url = f'https://api.worldbank.org/v2/country/{";".join(paises)}/indicator/{ind}?date=2010:2024&format=json'
#    response = requests.get(url)
#    print(response.json())


def feth_data_indicador (codigo_pais, id_indicador, fecha_inicio=2010, fecha_fin=2024):

    paises = ";".join(codigo_pais)
    endpoint = f'https://api.worldbank.org/v2/country/{paises}/indicator/{id_indicador}'
    pagina = 1
    datos_extendidos = []

    while True:
        params = {
            'format' : 'json',
            'date' : f'{fecha_inicio}:{fecha_fin}',
            'page' : pagina
        }
        response = requests.get(endpoint, params=params)
        data = response.json()

        if not isinstance(data,list) or len(data)==0:
            raise ValueError(f'Respuesta API no esperada para{id_indicador}:{data}')
    
        metadatos = data[0]
        datos = data[1]
        datos_extendidos.extend(datos)

        total_paginas = metadatos.get('pages')

        if pagina >= total_paginas:
            break
        pagina+=1
    
    return datos_extendidos



In [26]:
tablas = {}

for id_indicador, nombre_tabla in indicadores.items():
    data_raw = feth_data_indicador(paises,id_indicador)

    records = []
    for fila in data_raw:
        records.append({
            'pais': fila['country']['value'],
            'agno' : fila['date'],
            'valor': fila['value']
        })
    
    import pandas as pd

    df = pd.DataFrame(records)

    print(df)

             pais  agno       valor
0          Angola  2024    37885849
1          Angola  2023    36749906
2          Angola  2022    35635029
3          Angola  2021    34532429
4          Angola  2020    33451132
5          Angola  2019    32375632
6          Angola  2018    31297155
7          Angola  2017    30234839
8          Angola  2016    29183070
9          Angola  2015    28157798
10         Angola  2014    27160769
11         Angola  2013    26165620
12         Angola  2012    25177394
13         Angola  2011    24218352
14         Angola  2010    23294825
15          China  2024  1408975000
16          China  2023  1410710000
17          China  2022  1412175000
18          China  2021  1412360000
19          China  2020  1411100000
20          China  2019  1407745000
21          China  2018  1402760000
22          China  2017  1396215000
23          China  2016  1387790000
24          China  2015  1379860000
25          China  2014  1371860000
26          China  2013  136